In [2]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

url = "https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/refs/heads/main/Ch04.%20Advanced%20Rag/Data/%ED%88%AC%EC%9E%90%EC%84%A4%EB%AA%85%EC%84%9C.pdf"

loader = PyPDFLoader(url)

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

docs = loader.load_and_split(doc_splitter)

/Users/choyoungrae/Desktop/book-study/books/RAG 마스터 랭체인으로 완성하는 LLM 서비스/rag-master-practice-260326/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_openai.embeddings import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [5]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(docs, embedding)

In [7]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from typing import List, Dict, Any, Tuple
from langchain_openai import ChatOpenAI
from textwrap import dedent
from langchain_core.output_parsers import JsonOutputParser

In [8]:
class RelevanceScore(BaseModel):
    relevance_score: float = Field(description="문서가 쿼리와 얼마나 관련이 있는지를 나타내는 점수.")

def reranking_documents(query: str, docs: List[Document], top_n: int = 2) -> List[Document]:
    parser = JsonOutputParser(pydantic_object=RelevanceScore)
    human_message_prompt = PromptTemplate(
        template = """
        1점부터 10점까지 점수를 매겨, 다음 문서가 질문이 얼마나 관련이 있는지 평가해주세요. 단순히 키워드가 일치하는 것이 아니라 쿼리의 구체적인 맥락과 의도를 고려하세요.
        {format_instructions}
        question: {query}
        document: {doc}
        relevance_score:""",
        input_variables=["query", "doc"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )

    llm = ChatOpenAI(temperature=0, model="gpt-4o", max_tokens=3000)
    chain = human_message_prompt | llm | parser
    scored_docs = []
    for doc in docs:
        input_data = {"query": query, "doc": doc.page_content}
        try:
            score = chain.invoke(input_data)['relevance_score']
            score = float(score)
        except Exception as e:
            print(f"오류 발생: {str(e)}")
            default_score = 5  # 기본 점수를 5점으로 설정
            print(f"기본 점수 {default_score}점을 사용합니다.")
            score = default_score
        scored_docs.append((doc, score))

    reranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in reranked_docs[:top_n]]

In [9]:
query = "이 회사의 2022년 영업손실이 정확히 얼마야?"
initial_docs = faiss_store.similarity_search(query, k=4)
reranked_docs = reranking_documents(query, initial_docs)

In [10]:
# print first 4 initial documents
print(f"Query: {query}\n\n")

print("Top initial documents:")
for i, doc in enumerate(initial_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)


# Print results
print("\n\nTop reranked documents:")
for i, doc in enumerate(reranked_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)  # Print first 200 characters of each document

Query: 이 회사의 2022년 영업손실이 정확히 얼마야?


Top initial documents:

Document 1:
이어, 당사는 2022년 GMP시설의 위탁생산계약 매출 약 4.8억원이 발생하였으나, 종업원급
여 43.9억원, 유무형자산상각비 25.3억원이 발생하였으며, 연구개발 및 임상시험 지속 과정
에서 지급수수료가 54.8억원 발생하는 등 총 영업비용 154억원이 발생하였습니다. 이에
2022년 영업손실 149억원이 발생하였으며, 한편, 2022년 5월 금융감독원에서 공표한 '전환
사채 콜옵션 회계처리' 감독지침에 따라 당사는 2021년 3월 19일 발행한 제2회 전환사채의
콜옵션을 별도로 분리하여 회계처리 하는 것으로 수정하였고, 투자자의 조기상환청구권에
대해서도 별도로 구분하여 파생상품부채로 회계처리 한 결과, 전환사채의 전환가격과 주가
간 차이로 인해 파생상품 평가손실이 약 68억원 발생하였고, 이자비용 16.3억원이 발생한 점
이 주요 원인으로 작용하여 당기순손실 228.7억원이 발생하였습니다. 
 
이어, 당사는 2023년에는 매출이 발생하지 않았으며, 종업원급여 40.9억원, 유무형자산상각
비 26.3억원이 발생하였으며, 연구개발 및 임상시험 지속 과정에서 지급수수료가 30.5억원
발생하는 등 총 영업비용 122억원이 발생하였습니다. 이에 2023년 영업손실 122억원이 발
생하였으며, 금융손익 및 기타손익 반영 후 당기순손실 116.1억원이 발생하였습니다. 한편,
2023년말 당사 보유 토지에 대한 재평가가 이루어졌으며, 세금효과를 고려한 후의 재평가이
익 119.3억원이 기타포괄손익으로 인식되며 2023년에는 일시적으로 총포괄이익 3.1억원을
인식하였습니다. 
 
이어, 당사는 2024년 1분기에는 GMP시설의 위탁생산계약 매출 약 3.2억원이 발생하였으나,
종업원급여 9.5억원, 유무형자산상각비 6.6억원이 발생하였으며, 연구개발 및 임상시험 지
속 과정에서 지급수수료가 6.5억원 발생하는 등 총 영업비용 27.3억원이 발생